In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
# Task 1: Write your code here:
data_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(data_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# Price distribution (target variable)
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=( 'Order_ID'))
df.head()

In [ ]:
# Task 2: Write your code here:

# Do we have missing values? cheaack first
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)


In [ ]:
# handeling missing values
# Drop rows where target (Delivery_Time) or key features are missing - can't predict without them
df = df.dropna(subset=['Delivery_Time'])

# Fill with mode for categorical features
df['Weather'] = df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level'] = df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day'] = df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])

# Fill with mean for numerical features
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].median())

check_missing_values(df)

In [ ]:
# Task 3: Write your code here:
#Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
# select categorical columns first
categorical_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))

# one hot encoding
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
for col in categorical_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col])

df

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:
# REGRESSION TARGETS CAN'T BE IMBALANCED

In [ ]:
# Task 1: Write your code here:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import KFold
n_splits = 5  # K=5 Folds
# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=200)
MAE_losses = []



# training
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]
  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)
  # Calculate metric
  mae = mean_absolute_error(y_test, y_pred)
  # store results
  MAE_losses.append(mae)

print(f"  MAE Average loss:  {np.mean(MAE_losses):.4f}")





In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
})
importance = importance.sort_values('importance', ascending=True).tail(10)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'])
plt.title('Top 10 Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here:

valid_pred = model.predict(X)

plt.figure(figsize=(10, 6))
plt.hist(valid_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Emission')
plt.ylabel('Count')
plt.show()